## AutoGen Studio Agent Workflow API Example

This notebook focuses on demonstrating capabilities of the autogen studio workflow python api.  

- Declarative Specification of an Agent Team
- Loading the specification and running the resulting agent

 

In [ ]:
! export OPENAI_API_KEY="sk"

In [ ]:
from autogenstudio.teammanager import TeamManager
from autogen_agentchat.ui import Console

task ="""Please review the meidcal record and provide treatment plan: Patient Medical Record - Lung Cancer Patient ID: P123456 Name: John Doe Age: 67 Sex: Male DOB: 1957-03-15 Medical Record Number (MRN): MRN-789456
Chief Complaint Persistent cough, weight loss, and dyspnea (shortness of breath) for the past 3 months. Occasional chest pain and hemoptysis (coughing up blood). History of Present Illness (HPI) Onset: Symptoms started gradually 3 months ago. Progression: Increased severity in the last 4 weeks, prompting consultation. Smoking History: 40 pack-years, quit 5 years ago. Occupational Exposure: Worked in construction, possible asbestos exposure. Past Medical History (PMH) Chronic Obstructive Pulmonary Disease (COPD) Hypertension Hyperlipidemia Family History Father: Died from lung cancer at age 72. Mother: Hypertension, died at age 80. Siblings: No known cancer history. Medications Amlodipine 10 mg PO daily (for hypertension) Atorvastatin 20 mg PO daily (for hyperlipidemia) Albuterol inhaler PRN (for COPD) Allergies No known drug allergies (NKDA). Physical Examination General: Cachectic appearance, mild respiratory distress. Vitals: BP: 130/85 mmHg | HR: 88 bpm | RR: 22 | Temp: 98.6°F | SpO₂: 92% on room air. Lungs: Decreased breath sounds in the right upper lobe, dullness to percussion. Heart: Regular rate and rhythm, no murmurs. Imaging & Diagnostics Chest X-ray: Right upper lobe opacity with suspected mass. CT Scan (Chest): 4.5 cm mass in the right upper lobe, irregular borders. Enlarged mediastinal lymph nodes (suggestive of metastasis). No pleural effusion. PET Scan: High uptake in the right lung mass and mediastinal lymph nodes. No distant metastases detected. Bronchoscopy with Biopsy: Pathology confirms non-small cell lung cancer (NSCLC), adenocarcinoma subtype. EGFR mutation: Negative. ALK rearrangement: Negative. PD-L1 expression: 50% (eligible for immunotherapy). Staging (TNM Classification) Tumor (T): T2b (≥4 cm, no invasion of major structures). Nodes (N): N2 (mediastinal lymph node involvement). Metastasis (M): M0 (no distant metastases). Final Staging: Stage IIIA NSCLC"""
wm = TeamManager()
result = await wm.run(task=task, team_config="team.json")


In [ ]:
result_stream =  wm.run_stream(task=task, team_config="team.json")
async for response in result_stream:
    print(response)

## Load Directly Using the AgentChat API 

In [ ]:
import json 
from autogen_agentchat.teams import BaseGroupChat
team_config = json.load(open("team.json"))  
team = BaseGroupChat.load_component(team_config)
print(team._participants)

## AutoGen Studio Database API

Api for creating objects and serializing to a database.

In [ ]:
from autogenstudio.database import DatabaseManager
import os
# delete database
# if os.path.exists("test.db"):
#     os.remove("test.db")

os.makedirs("test", exist_ok=True)
# create a database
dbmanager = DatabaseManager(engine_uri="sqlite:///test.db", base_dir="test")
dbmanager.initialize_database()

## Sample AgentChat Example (Python)

In [ ]:
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.conditions import TextMentionTermination, MaxMessageTermination
from autogen_agentchat.teams import RoundRobinGroupChat, SelectorGroupChat
from autogen_ext.models.openai import OpenAIChatCompletionClient

planner_agent = AssistantAgent(
    "planner_agent",
    model_client=OpenAIChatCompletionClient(model="gpt-4"),
    description="A helpful assistant that can plan trips.",
    system_message="You are a helpful assistant that can suggest a travel plan for a user based on their request. Respond with a single sentence",
)

local_agent = AssistantAgent(
    "local_agent",
    model_client=OpenAIChatCompletionClient(model="gpt-4"),
    description="A local assistant that can suggest local activities or places to visit.",
    system_message="You are a helpful assistant that can suggest authentic and interesting local activities or places to visit for a user and can utilize any context information provided. Respond with a single sentence",
)

language_agent = AssistantAgent(
    "language_agent",
    model_client=OpenAIChatCompletionClient(model="gpt-4"),
    description="A helpful assistant that can provide language tips for a given destination.",
    system_message="You are a helpful assistant that can review travel plans, providing feedback on important/critical tips about how best to address language or communication challenges for the given destination. If the plan already includes language tips, you can mention that the plan is satisfactory, with rationale.Respond with a single sentence",
)

travel_summary_agent = AssistantAgent(
    "travel_summary_agent",
    model_client=OpenAIChatCompletionClient(model="gpt-4"),
    description="A helpful assistant that can summarize the travel plan.",
    system_message="You are a helpful assistant that can take in all of the suggestions and advice from the other agents and provide a detailed tfinal travel plan. You must ensure th b at the final plan is integrated and complete. YOUR FINAL RESPONSE MUST BE THE COMPLETE PLAN. When the plan is complete and all perspectives are integrated, you can respond with TERMINATE.Respond with a single sentence",
)

termination = TextMentionTermination("TERMINATE") | MaxMessageTermination(10)
group_chat = RoundRobinGroupChat(
    [planner_agent, local_agent, language_agent, travel_summary_agent], termination_condition=termination
)




In [ ]:
result = group_chat.run_stream(task="Plan a 3 day trip to Nepal.")
async for response in result:
    print(response)

In [ ]:
import json

# convert to config 
config = group_chat.dump_component().model_dump()
# save as json 

with open("travel_team.json", "w") as f:
    json.dump(config, f, indent=4)

# load from json
with open("travel_team.json", "r") as f:
    config = json.load(f)

group_chat = RoundRobinGroupChat.load_component(config) 
result = group_chat.run_stream(task="Plan a 3 day trip to Nepal.") 
async for response in result:
    print(response)